# Baseline comparison summaries

This notebook reads the persisted Main9 experiment artifacts directly from `comp/` and `stability/`. For every seed or HDBSCAN parameter setting, it aligns the nine-point experimental curve with the deterministic full-PCA baseline before computing two-sided Spearman correlation, MAE, and $R^2$. The displayed values are the cross-unit mean $\pm$ sample standard deviation (`ddof=1`). Mean experimental curves are never used for these summaries.

O-S uses `global_mean`. MS-E uses raw `mean_H`.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, r2_score


def find_project_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        required = (candidate / "README.md", candidate / "comp", candidate / "stability")
        if required[0].is_file() and required[1].is_dir() and required[2].is_dir():
            return candidate
    raise FileNotFoundError(f"Could not locate the project root from {start}")


PROJECT_ROOT = find_project_root(Path.cwd())
MAIN9_POINT_COUNT = 9

MODEL_SPECS = (
    {
        "label": "Mistral-7B",
        "baseline_dir": Path("comp/mistralai/Mistral-7B-v0.1/output_proj"),
        "global_permutation_dir": Path("comp/permutation/mistralai/Mistral-7B-v0.1_rand/output_proj"),
        "length_bucket_dir": Path("comp/permutation/mistralai/Mistral-7B-v0.1_length_bucket_rand/output_proj"),
        "stability_dir": Path("mistralai/Mistral-7B-v0.1/output_proj"),
    },
    {
        "label": "Mixtral-8x7B",
        "baseline_dir": Path("comp/mistralai/Mixtral-8x7B-v0.1/output_proj"),
        "global_permutation_dir": Path("comp/permutation/mistralai/Mixtral-8x7B-v0.1_rand/output_proj"),
        "length_bucket_dir": Path("comp/permutation/mistralai/Mixtral-8x7B-v0.1_length_bucket_rand/output_proj"),
        "stability_dir": Path("mistralai/Mixtral-8x7B-v0.1/output_proj"),
    },
    {
        "label": "GPT-oss-20B",
        "baseline_dir": Path("comp/gpt-oss/output_proj"),
        "global_permutation_dir": Path("comp/permutation/gpt-oss_rand/output_proj"),
        "length_bucket_dir": Path("comp/permutation/gpt-oss_length_bucket_rand/output_proj"),
        "stability_dir": Path("gpt-oss/output_proj"),
    },
)

METRIC_SPECS = {
    "O-S": {
        "baseline_file": "kondrak_global_summary.csv",
        "value_column": "global_mean",
    },
    "MS-E": {
        "baseline_file": "script_entropy_summary.csv",
        "value_column": "mean_H",
    },
}

HDBSCAN_SETTINGS = (
    "mcs=6_ms=5",
    "mcs=5_ms=6",
    "mcs=6_ms=6",
    "mcs=4_ms=5",
    "mcs=5_ms=4",
    "mcs=4_ms=4",
)

print(f"Project root: {PROJECT_ROOT}")

Project root: /home/void/Projects/EmbdAlys


In [2]:
def require_file(relative_path):
    path = PROJECT_ROOT / relative_path
    if not path.is_file():
        raise FileNotFoundError(f"Required data file does not exist: {path}")
    return path


def read_metric_frame(relative_path, value_column, unit_column=None):
    path = require_file(relative_path)
    frame = pd.read_csv(path)
    required_columns = ["pca_dim", value_column]
    if unit_column is not None:
        required_columns.insert(0, unit_column)
    missing_columns = [column for column in required_columns if column not in frame.columns]
    if missing_columns:
        raise ValueError(f"{path} is missing columns: {missing_columns}")

    output = frame.loc[:, required_columns].copy()
    output["pca_dim"] = pd.to_numeric(output["pca_dim"], errors="raise").astype(int)
    output[value_column] = pd.to_numeric(output[value_column], errors="raise").astype(float)
    if output[value_column].isna().any():
        raise ValueError(f"{path} contains missing values in {value_column}")
    if unit_column is not None:
        output[unit_column] = pd.to_numeric(output[unit_column], errors="raise").astype(int)
    return output


def load_baseline_curve(model_spec, metric):
    metric_spec = METRIC_SPECS[metric]
    relative_path = model_spec["baseline_dir"] / metric_spec["baseline_file"]
    frame = read_metric_frame(relative_path, metric_spec["value_column"])
    if len(frame) != MAIN9_POINT_COUNT:
        raise ValueError(f"Expected {MAIN9_POINT_COUNT} baseline points in {relative_path}, found {len(frame)}")
    if frame["pca_dim"].duplicated().any():
        raise ValueError(f"Duplicate baseline pca_dim values in {relative_path}")
    return frame.rename(columns={metric_spec["value_column"]: "baseline_value"}).sort_values("pca_dim")


def compare_with_baseline(baseline_curve, experimental_curve, source_label):
    if experimental_curve["pca_dim"].duplicated().any():
        raise ValueError(f"Duplicate experimental pca_dim values in {source_label}")

    aligned = baseline_curve.merge(
        experimental_curve,
        on="pca_dim",
        how="outer",
        validate="one_to_one",
        indicator=True,
    )
    unmatched = aligned.loc[aligned["_merge"] != "both", ["pca_dim", "_merge"]]
    if not unmatched.empty:
        raise ValueError(f"Main9 scale mismatch in {source_label}: {unmatched.to_dict('records')}")
    if len(aligned) != MAIN9_POINT_COUNT:
        raise ValueError(f"Expected {MAIN9_POINT_COUNT} aligned points in {source_label}, found {len(aligned)}")

    aligned = aligned.sort_values("pca_dim")
    y_true = aligned["baseline_value"].to_numpy(dtype=float)
    y_pred = aligned["experimental_value"].to_numpy(dtype=float)
    spearman_result = spearmanr(y_true, y_pred, alternative="two-sided")
    return {
        "n_scales": int(len(aligned)),
        "spearman_rho": float(spearman_result.statistic),
        "spearman_p": float(spearman_result.pvalue),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true=y_true, y_pred=y_pred)),
    }


def compute_seeded_experiment(
    experiment_name,
    directory_builder,
    file_by_metric,
    seed_column,
    metrics,
    expected_seed_count,
):
    records = []
    for model_spec in MODEL_SPECS:
        reference_seeds = None
        for metric in metrics:
            metric_spec = METRIC_SPECS[metric]
            relative_path = directory_builder(model_spec) / file_by_metric[metric]
            frame = read_metric_frame(relative_path, metric_spec["value_column"], seed_column)
            seeds = tuple(sorted(frame[seed_column].unique().tolist()))
            if len(seeds) != expected_seed_count:
                raise ValueError(
                    f"Expected {expected_seed_count} seeds for {experiment_name}, "
                    f"{model_spec['label']}, {metric}; found {len(seeds)}"
                )
            if reference_seeds is None:
                reference_seeds = seeds
            elif seeds != reference_seeds:
                raise ValueError(f"Seed mismatch across metrics for {experiment_name}, {model_spec['label']}")

            baseline_curve = load_baseline_curve(model_spec, metric)
            for seed in seeds:
                seed_curve = (
                    frame.loc[frame[seed_column] == seed, ["pca_dim", metric_spec["value_column"]]]
                    .rename(columns={metric_spec["value_column"]: "experimental_value"})
                    .sort_values("pca_dim")
                )
                source_label = f"{experiment_name}, {model_spec['label']}, {metric}, {seed_column}={seed}"
                stats = compare_with_baseline(baseline_curve, seed_curve, source_label)
                records.append({
                    "experiment": experiment_name,
                    "model": model_spec["label"],
                    "metric": metric,
                    "unit": int(seed),
                    **stats,
                })
    return pd.DataFrame.from_records(records)


def compute_hdbscan_stability():
    records = []
    for model_spec in MODEL_SPECS:
        for metric in ("O-S", "MS-E"):
            metric_spec = METRIC_SPECS[metric]
            baseline_curve = load_baseline_curve(model_spec, metric)
            for setting in HDBSCAN_SETTINGS:
                relative_path = (
                    Path("stability")
                    / setting
                    / model_spec["stability_dir"]
                    / metric_spec["baseline_file"]
                )
                setting_curve = (
                    read_metric_frame(relative_path, metric_spec["value_column"])
                    .rename(columns={metric_spec["value_column"]: "experimental_value"})
                    .sort_values("pca_dim")
                )
                source_label = f"HDBSCAN stability, {model_spec['label']}, {metric}, {setting}"
                stats = compare_with_baseline(baseline_curve, setting_curve, source_label)
                records.append({
                    "experiment": "HDBSCAN stability",
                    "model": model_spec["label"],
                    "metric": metric,
                    "unit": setting,
                    **stats,
                })
    return pd.DataFrame.from_records(records)


def summarize_comparisons(per_unit_frame, metrics, expected_unit_count):
    records = []
    for model_spec in MODEL_SPECS:
        for metric in metrics:
            subset = per_unit_frame.loc[
                (per_unit_frame["model"] == model_spec["label"])
                & (per_unit_frame["metric"] == metric)
            ]
            if len(subset) != expected_unit_count:
                raise ValueError(
                    f"Expected {expected_unit_count} comparison rows for "
                    f"{model_spec['label']}, {metric}; found {len(subset)}"
                )
            records.append({
                "Model": model_spec["label"],
                "Metrics": metric,
                "rho_mean": subset["spearman_rho"].mean(),
                "rho_sd": subset["spearman_rho"].std(ddof=1),
                "p_mean": subset["spearman_p"].mean(),
                "p_sd": subset["spearman_p"].std(ddof=1),
                "mae_mean": subset["mae"].mean(),
                "mae_sd": subset["mae"].std(ddof=1),
                "r2_mean": subset["r2"].mean(),
                "r2_sd": subset["r2"].std(ddof=1),
            })
    return pd.DataFrame.from_records(records)


def format_mean_sd(mean_value, sd_value):
    return f"{mean_value:.6g} ± {sd_value:.6g}"


def format_summary_table(summary_frame):
    return pd.DataFrame({
        "Model": summary_frame["Model"],
        "Metrics": summary_frame["Metrics"],
        r"Spearman $\rho$": [
            format_mean_sd(mean_value, sd_value)
            for mean_value, sd_value in zip(summary_frame["rho_mean"], summary_frame["rho_sd"])
        ],
        r"$p$-value": [
            format_mean_sd(mean_value, sd_value)
            for mean_value, sd_value in zip(summary_frame["p_mean"], summary_frame["p_sd"])
        ],
        "MAE": [
            format_mean_sd(mean_value, sd_value)
            for mean_value, sd_value in zip(summary_frame["mae_mean"], summary_frame["mae_sd"])
        ],
        r"$R^2$": [
            format_mean_sd(mean_value, sd_value)
            for mean_value, sd_value in zip(summary_frame["r2_mean"], summary_frame["r2_sd"])
        ],
    })


def display_summary(summary_frame):
    display(format_summary_table(summary_frame).style.hide(axis="index"))

## Randomized PCA

Eight `pca_seed` curves per model and metric are read from each baseline model directory under `output_proj/random/`.

In [3]:
randomized_pca_per_seed = compute_seeded_experiment(
    experiment_name="Randomized PCA",
    directory_builder=lambda model_spec: model_spec["baseline_dir"] / "random",
    file_by_metric={
        "O-S": "kondrak_global_summary_all_seeds.csv",
        "MS-E": "script_entropy_all_seeds.csv",
    },
    seed_column="pca_seed",
    metrics=("O-S", "MS-E"),
    expected_seed_count=8,
)
randomized_pca_summary = summarize_comparisons(
    randomized_pca_per_seed,
    metrics=("O-S", "MS-E"),
    expected_unit_count=8,
)
display_summary(randomized_pca_summary)

Model,Metrics,Spearman $\rho$,$p$-value,MAE,$R^2$
Mistral-7B,O-S,0.964583 ± 0.0326568,0.000138576 ± 0.000229095,0.00415362 ± 0.000904102,0.996082 ± 0.00226218
Mistral-7B,MS-E,0.960417 ± 0.0307802,0.000125916 ± 0.00017259,0.00440421 ± 0.00143323,0.985075 ± 0.00995495
Mixtral-8x7B,O-S,0.95625 ± 0.0526575,0.000522417 ± 0.000969613,0.002503 ± 0.00125917,0.998027 ± 0.00225987
Mixtral-8x7B,MS-E,0.929167 ± 0.0532663,0.000883561 ± 0.00126598,0.00257595 ± 0.00122199,0.981093 ± 0.0200827
GPT-oss-20B,O-S,0.995833 ± 0.00771517,4.84049e-07 ± 8.96285e-07,0.00279635 ± 0.00104147,0.997261 ± 0.00269299
GPT-oss-20B,MS-E,0.964583 ± 0.054509,0.000664325 ± 0.00185926,0.00423762 ± 0.00146665,0.993072 ± 0.00575032


## HDBSCAN stability

Each of the six persisted HDBSCAN parameter settings is compared separately with the deterministic full-PCA baseline.

In [4]:
hdbscan_stability_per_setting = compute_hdbscan_stability()
hdbscan_stability_summary = summarize_comparisons(
    hdbscan_stability_per_setting,
    metrics=("O-S", "MS-E"),
    expected_unit_count=len(HDBSCAN_SETTINGS),
)
display_summary(hdbscan_stability_summary)

Model,Metrics,Spearman $\rho$,$p$-value,MAE,$R^2$
Mistral-7B,O-S,0.941667 ± 0.0491596,0.000686633 ± 0.00148105,0.0126301 ± 0.00455639,0.984458 ± 0.0107393
Mistral-7B,MS-E,0.797222 ± 0.211454,0.0477018 ± 0.0746701,0.0110383 ± 0.00408323,0.915122 ± 0.102219
Mixtral-8x7B,O-S,0.686111 ± 0.15826,0.0698509 ± 0.0922187,0.0129993 ± 0.00495629,0.981324 ± 0.0125744
Mixtral-8x7B,MS-E,0.888889 ± 0.140502,0.011441 ± 0.0231546,0.0102161 ± 0.00355027,0.878134 ± 0.0646389
GPT-oss-20B,O-S,0.980556 ± 0.0245327,4.02847e-05 ± 9.58361e-05,0.0164786 ± 0.00629552,0.965502 ± 0.0257758
GPT-oss-20B,MS-E,0.919444 ± 0.0805651,0.0024271 ± 0.0037164,0.0223073 ± 0.00810299,0.923453 ± 0.0522556


## Global token permutation

Ten `perm_seed` curves per model and metric are read from the global-permutation `output_proj` directories.

In [5]:
global_permutation_per_seed = compute_seeded_experiment(
    experiment_name="Global token permutation",
    directory_builder=lambda model_spec: model_spec["global_permutation_dir"],
    file_by_metric={
        "O-S": "kondrak_global_summary_all_seeds.csv",
        "MS-E": "script_entropy_summary_all_seeds.csv",
    },
    seed_column="perm_seed",
    metrics=("O-S", "MS-E"),
    expected_seed_count=10,
)
global_permutation_summary = summarize_comparisons(
    global_permutation_per_seed,
    metrics=("O-S", "MS-E"),
    expected_unit_count=10,
)
display_summary(global_permutation_summary)

Model,Metrics,Spearman $\rho$,$p$-value,MAE,$R^2$
Mistral-7B,O-S,0.328333 ± 0.00805076,0.388418 ± 0.0124065,0.358717 ± 0.000922021,-9.13625 ± 0.0418982
Mistral-7B,MS-E,-0.183333 ± 0.0647884,0.639485 ± 0.125892,0.198126 ± 0.00372482,-15.0653 ± 1.01787
Mixtral-8x7B,O-S,-0.0733333 ± 0.271734,0.5199 ± 0.152125,0.338218 ± 0.000721839,-8.80861 ± 0.0253302
Mixtral-8x7B,MS-E,-0.41 ± 0.099132,0.287028 ± 0.123514,0.16766 ± 0.00300449,-36.9717 ± 1.44048
GPT-oss-20B,O-S,-0.416667 ± 0.053287,0.268635 ± 0.067945,0.396054 ± 9.31512e-05,-16.1506 ± 0.00619026
GPT-oss-20B,MS-E,-0.865 ± 0.061589,0.0046952 ± 0.00628199,0.158325 ± 0.00101268,-2.94392 ± 0.0601582


## Length-bucket token permutation

This control is defined only for O-S. Ten `perm_seed` O-S curves per model are read from the length-bucket permutation directories.

In [6]:
length_bucket_per_seed = compute_seeded_experiment(
    experiment_name="Length-bucket token permutation",
    directory_builder=lambda model_spec: model_spec["length_bucket_dir"],
    file_by_metric={"O-S": "kondrak_global_summary_all_seeds.csv"},
    seed_column="perm_seed",
    metrics=("O-S",),
    expected_seed_count=10,
)
length_bucket_summary = summarize_comparisons(
    length_bucket_per_seed,
    metrics=("O-S",),
    expected_unit_count=10,
)
display_summary(length_bucket_summary)

Model,Metrics,Spearman $\rho$,$p$-value,MAE,$R^2$
Mistral-7B,O-S,0.545 ± 0.117864,0.149082 ± 0.0950629,0.351433 ± 0.000760017,-8.7701 ± 0.0296941
Mixtral-8x7B,O-S,0.675 ± 0.110624,0.0607675 ± 0.0540359,0.33451 ± 0.000417134,-8.41576 ± 0.0131735
GPT-oss-20B,O-S,-0.373333 ± 0.052234,0.326045 ± 0.0734953,0.38963 ± 0.00021207,-15.6177 ± 0.0177535


## Random orthogonal projection

Ten `rp_seed` curves per model and metric are read from each baseline model directory under `output_proj/random_projection/`.

In [7]:
random_projection_per_seed = compute_seeded_experiment(
    experiment_name="Random orthogonal projection",
    directory_builder=lambda model_spec: model_spec["baseline_dir"] / "random_projection",
    file_by_metric={
        "O-S": "kondrak_global_summary_all_seeds.csv",
        "MS-E": "script_entropy_summary_all_seeds.csv",
    },
    seed_column="rp_seed",
    metrics=("O-S", "MS-E"),
    expected_seed_count=10,
)
random_projection_summary = summarize_comparisons(
    random_projection_per_seed,
    metrics=("O-S", "MS-E"),
    expected_unit_count=10,
)
display_summary(random_projection_summary)

Model,Metrics,Spearman $\rho$,$p$-value,MAE,$R^2$
Mistral-7B,O-S,0.0483333 ± 0.254763,0.586766 ± 0.203954,0.026988 ± 0.000777122,0.809845 ± 0.0119292
Mistral-7B,MS-E,0.591667 ± 0.171279,0.133062 ± 0.123291,0.0247773 ± 0.000948203,-0.0500984 ± 0.095162
Mixtral-8x7B,O-S,0.105 ± 0.285022,0.574101 ± 0.306655,0.0113812 ± 0.000621121,0.968109 ± 0.00196176
Mixtral-8x7B,MS-E,0.126667 ± 0.25058,0.588305 ± 0.268391,0.0512746 ± 0.00187698,-10.5623 ± 1.06751
GPT-oss-20B,O-S,0.318333 ± 0.263693,0.473212 ± 0.349175,0.0284571 ± 0.000689298,0.737377 ± 0.0230825
GPT-oss-20B,MS-E,0.24 ± 0.360144,0.410852 ± 0.32654,0.137674 ± 0.00458857,-5.33281 ± 0.756154


## Copyable combined summary

The following cell prints all experiment tables in one Markdown text block.

In [8]:
def dataframe_to_markdown(frame):
    headers = [str(column) for column in frame.columns]
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    for row in frame.itertuples(index=False, name=None):
        values = [str(value).replace("|", r"\|") for value in row]
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)


combined_summaries = (
    ("Randomized PCA", randomized_pca_summary),
    ("HDBSCAN stability", hdbscan_stability_summary),
    ("Global token permutation", global_permutation_summary),
    ("Length-bucket token permutation", length_bucket_summary),
    ("Random orthogonal projection", random_projection_summary),
)

for experiment_name, summary_frame in combined_summaries:
    print(f"## {experiment_name}")
    print(dataframe_to_markdown(format_summary_table(summary_frame)))
    print()

## Randomized PCA
| Model | Metrics | Spearman $\rho$ | $p$-value | MAE | $R^2$ |
| --- | --- | --- | --- | --- | --- |
| Mistral-7B | O-S | 0.964583 ± 0.0326568 | 0.000138576 ± 0.000229095 | 0.00415362 ± 0.000904102 | 0.996082 ± 0.00226218 |
| Mistral-7B | MS-E | 0.960417 ± 0.0307802 | 0.000125916 ± 0.00017259 | 0.00440421 ± 0.00143323 | 0.985075 ± 0.00995495 |
| Mixtral-8x7B | O-S | 0.95625 ± 0.0526575 | 0.000522417 ± 0.000969613 | 0.002503 ± 0.00125917 | 0.998027 ± 0.00225987 |
| Mixtral-8x7B | MS-E | 0.929167 ± 0.0532663 | 0.000883561 ± 0.00126598 | 0.00257595 ± 0.00122199 | 0.981093 ± 0.0200827 |
| GPT-oss-20B | O-S | 0.995833 ± 0.00771517 | 4.84049e-07 ± 8.96285e-07 | 0.00279635 ± 0.00104147 | 0.997261 ± 0.00269299 |
| GPT-oss-20B | MS-E | 0.964583 ± 0.054509 | 0.000664325 ± 0.00185926 | 0.00423762 ± 0.00146665 | 0.993072 ± 0.00575032 |

## HDBSCAN stability
| Model | Metrics | Spearman $\rho$ | $p$-value | MAE | $R^2$ |
| --- | --- | --- | --- | --- | --- |
| Mistral-7B | O-S | 